# Qwen2.5 all-baselines evaluation


In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv>=1,<2"], check=True)
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get("CRASHDIAG_ENV_FILE", LAUNCH_DIR / ".env")).expanduser()
if not ENV_FILE.is_absolute():
    ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if ENV_FILE.is_file():
    load_dotenv(ENV_FILE, override=True)

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

KAGGLE_SECRET_ALIASES = {
    "HF_TOKEN": "HF_TOKEN",
    "CRASHDIAG_DATASET_RUN_ID": "CRASHDIAG_DATASET_RUN_ID",
    "DATASET_RUN_ID": "CRASHDIAG_DATASET_RUN_ID",
    "CRASHDIAG_SANDBOX_URL": "CRASHDIAG_SANDBOX_URL",
    "CRASHDIAG_API_TOKEN": "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_SANDBOX_TOKEN": "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_SOURCE_COMMIT": "CRASHDIAG_SOURCE_COMMIT",
    "SOURCE_COMMIT": "CRASHDIAG_SOURCE_COMMIT",

}
loaded_kaggle_secrets = []
kaggle_secret_errors = {}
if UserSecretsClient is not None:
    secrets = UserSecretsClient()
    for secret_name, env_name in KAGGLE_SECRET_ALIASES.items():
        if os.environ.get(env_name):
            continue
        try:
            value = secrets.get_secret(secret_name)
        except Exception as exc:
            kaggle_secret_errors[secret_name] = f"{type(exc).__name__}: {exc}"
            continue
        if value:
            os.environ[env_name] = value
            loaded_kaggle_secrets.append(secret_name)
print("loaded Kaggle secret names:", loaded_kaggle_secrets or "none")

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path(os.environ.get("CRASHDIAG_WORKDIR", LAUNCH_DIR / "CrashDiag-runtime")).expanduser().resolve()
if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f"CRASHDIAG_WORKDIR exists and is not a Git checkout: {WORKDIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print(f"env_file={ENV_FILE if ENV_FILE.is_file() else 'not present (using runtime/Kaggle secrets)'}")
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BUCKET_ID = "devaanshpa/CrashDiag"
MODELS = {
    "qwen2.5_14b": "Qwen/Qwen2.5-14B-Instruct",
    "qwen2.5_7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen2.5_3b": "Qwen/Qwen2.5-3B-Instruct",
    "qwen2.5_1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen2.5_0.5b": "Qwen/Qwen2.5-0.5B-Instruct",
}
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage, slug):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{slug}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"models={list(MODELS)}")
print(f"dataset_run_id={DATASET_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

CURRICULUM = os.environ.get("CRASHDIAG_CURRICULUM", "hard-v4").strip().lower()
EVAL_FILE = "grpo_hard_eval.jsonl" if CURRICULUM == "hard-v4" else "grpo_eval.jsonl"
DATASET_DIR = Path("artifacts/datasets")
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("datasets", DATASET_DIR)
assert (DATASET_DIR / EVAL_FILE).is_file(), f"dataset stage missing {EVAL_FILE}; check CRASHDIAG_DATASET_RUN_ID={DATASET_RUN_ID}"
print(f"curriculum={CURRICULUM}")
print(f"eval_file={EVAL_FILE}")


In [ ]:
from training.evaluate_jsonl import main as evaluate_main

results = {}
for slug, base_model in MODELS.items():
    run_id = os.environ.get(f"CRASHDIAG_BASE_{slug.upper().replace('.', '_')}_RUN_ID") or ist_run_id("base-eval", slug)
    print(f"=== evaluating {base_model} ({slug}) -> {run_id} ===")
    exit_code = evaluate_main([
        "--model", base_model,
        "--dataset", str(DATASET_DIR / EVAL_FILE),
        "--output-dir", f"outputs/{slug}-base-eval",
        "--load-in-4bit",
        "--precision", "bf16",
        "--max-new-tokens", "64",
        "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
        "--artifact-bucket", BUCKET_ID,
        "--run-id", run_id,
        "--artifact-stage", "base-eval",
    ])
    if exit_code:
        raise RuntimeError(f"{base_model} baseline evaluation failed: {exit_code}")
    import json
    report = json.loads((Path(f"outputs/{slug}-base-eval") / "mechanical_evaluation.json").read_text(encoding="utf-8"))
    results[slug] = report["summary"]
    print(f"{slug}: {report['summary']}")

print("\n=== ALL BASELINE RESULTS ===")
for slug, summary in results.items():
    print(f"{slug}: success={summary['success_rate']:.1%} strict_json={summary['strict_json_rate']:.1%} backend_error={summary['backend_error_rate']:.1%}")


In [ ]:
# Model-wise comparison across all baselines
from pathlib import Path as _P
import json as _j, html as _h

slugs = list(MODELS)
_chart_rows = []
for _slug in slugs:
    _p2 = _P(f"outputs/{_slug}-base-eval/mechanical_evaluation.json")
    if not _p2.exists():
        print(f"missing: {_p2}"); continue
    _data = _j.loads(_p2.read_text())
    _s = _data["summary"]
    _chart_rows.append((_slug, _s["success_rate"], _s["strict_json_rate"], _s["backend_error_rate"], dict(_data.get("per_fault", {}))))

for _slug, _, _, _, _per_fault in _chart_rows:
    print(f"\n[{_slug}] per-fault:")
    for _fault in sorted(_per_fault):
        _v = _per_fault[_fault]
        print(f"  {_fault}: {_v['resolved']}/{_v['episodes']}  {_v['success_rate']:.1%}")

_width,_height,_left,_right,_top,_bottom = 960,480,78,24,70,120
_plot_w = _width - _left - _right
_plot_h = _height - _top - _bottom
_n = len(_chart_rows) or 1
_slot = _plot_w / _n
_bar = _slot * 0.34
_colors = {"success": "#2563eb", "strict_json": "#16a34a"}
_parts = [
    '<?xml version="1.0" encoding="UTF-8"?>',
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{_width}" height="{_height}" viewBox="0 0 {_width} {_height}" role="img">',
    '<rect width="100%" height="100%" fill="#ffffff"/>',
    f'<text x="{_width/2:.1f}" y="34" text-anchor="middle" font-family="sans-serif" font-size="22" font-weight="600">Baseline comparison by model (hard-v4)</text>',
]
for _t in range(6):
    _r = _t/5; _y = _top + (1-_r)*_plot_h
    _parts += [f'<line x1="{_left}" y1="{_y:.1f}" x2="{_width-_right}" y2="{_y:.1f}" stroke="#e5e7eb" stroke-width="1"/>',
               f'<text x="{_left-10}" y="{_y+4:.1f}" text-anchor="end" font-family="sans-serif" font-size="12" fill="#4b5563">{_r:.0%}</text>']
for _i,(_slug,_s,_strict,_err,_) in enumerate(_chart_rows):
    _x0 = _left + _i*_slot + (_slot - 2*_bar - 6)/2
    for _j,(_val,_c) in enumerate([(_s,_colors["success"]),(_strict,_colors["strict_json"])]):
        _bh = _val*_plot_h; _y = _top + _plot_h - _bh
        _parts += [f'<rect x="{_x0+_j*(_bar+6):.1f}" y="{_y:.1f}" width="{_bar:.1f}" height="{_bh:.1f}" fill="{_c}" rx="3"/>',
                   f'<text x="{_x0+_j*(_bar+6)+_bar/2:.1f}" y="{max(_top+14,_y-8):.1f}" text-anchor="middle" font-family="sans-serif" font-size="11" font-weight="600">{_val:.0%}</text>']
    _cx = _left + _i*_slot + _slot/2
    _parts += [f'<text x="{_cx:.1f}" y="{_height-_bottom+22}" text-anchor="end" transform="rotate(-30 {_cx:.1f} {_height-_bottom+22})" font-family="sans-serif" font-size="12">{_h.escape(_slug)}</text>']
_parts += [f'<line x1="{_left}" y1="{_top}" x2="{_left}" y2="{_height-_bottom}" stroke="#111827" stroke-width="1.2"/>',
           f'<line x1="{_left}" y1="{_height-_bottom}" x2="{_width-_right}" y2="{_height-_bottom}" stroke="#111827" stroke-width="1.2"/>',
           f'<rect x="{_left}" y="{55}" width="18" height="9" fill="{_colors["success"]}"/><text x="{_left+24}" y="{63}" font-family="sans-serif" font-size="12">success_rate</text>',
           f'<rect x="{_left+120}" y="{55}" width="18" height="9" fill="{_colors["strict_json"]}"/><text x="{_left+144}" y="{63}" font-family="sans-serif" font-size="12">strict_json_rate</text>',
           "</svg>"]
_svg = _P("outputs/baselines_summary.svg")
_svg.parent.mkdir(parents=True, exist_ok=True)
_svg.write_text("\n".join(_parts), encoding="utf-8")

from IPython.display import SVG, display
display(SVG(filename=str(_svg)))

for _slug, _s, _strict, _err, _ in _chart_rows:
    print(f"{_slug:14s}  success {_s:.1%}  strict_json {_strict:.1%}  backend_error {_err:.1%}")
